# 01 — Probe de un día

Valida CDS + ERA5-Land en un punto y un día. Editá `LAT`, `LON` y `DAY` en la celda de abajo y volvé a correr desde ahí.

**Cobertura (catálogo CDS, 2026-10-05):** 1950-01-01 hasta **2026-09-29**. Se actualiza a diario, unos 5 días detrás del tiempo real (ERA5-Land-T). El mes consolidado reemplaza esa versión unos 2 meses después. Es reanálisis, no un forecast.

**NaN en todas las horas:** ERA5-Land solo tiene valores en tierra. Si la celda de 0.1° cae en agua (fiordo, lago, mar), temperatura, viento, precipitación y radiación vienen en NaN en cualquier fecha. El punto (-41.425382, -73.095492) queda en tierra, al oeste de Puerto Montt.

**Antes:** aceptá la licencia de [ERA5-Land hourly](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land?tab=download) con la misma cuenta del token.

Credenciales: `CDS_API_KEY` en `.env` o `%USERPROFILE%\.cdsapirc`.

In [ ]:
from dataclasses import replace
from datetime import date
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ecmwf_ml.client import build_retrieve_request, retrieve
from ecmwf_ml.config import get_settings
from ecmwf_ml.extract import to_frame

# --- editá estos tres y reejecutá desde esta celda ---
LAT = -41.425382
LON = -73.095492
DAY = date(2026, 9, 29)  # última fecha del catálogo CDS al 2026-10-05
# -----------------------------------------------------

settings = replace(get_settings(), lat=LAT, lon=LON, probe_date=DAY)
day = settings.probe_date
settings.dataset, settings.area, day

## Request CDS

`area` es `[North, West, South, East]` (±0.05°). CDS recorta en el servidor a ~una celda de 0.1°.

In [ ]:
request = build_retrieve_request(day, day, settings)
request

## Descarga de un día

Si el NetCDF ya existe, no se vuelve a pedir. La cola de CDS puede tardar unos minutos.

In [ ]:
nc_path = retrieve(day, day, settings=settings)
size_kb = nc_path.stat().st_size / 1024
print(nc_path)
print(f"{size_kb:.1f} KiB")
assert size_kb < 20_000, "El archivo es demasiado grande; el recorte espacial probablemente falló."

## Inspección del NetCDF

In [ ]:
import xarray as xr

ds = xr.open_dataset(nc_path)
print(ds)
print("sizes:", dict(ds.sizes))

In [ ]:
frame = to_frame(nc_path, settings)
print(frame.shape)
display(frame.head(8))

assert 20 <= len(frame) <= 25, f"Se esperaban ~24 horas, hay {len(frame)}"

value_cols = [c for c in ("t2m_c", "u10", "v10", "tp_mm", "ssrd_wm2") if c in frame.columns]
if value_cols and bool(frame[value_cols].isna().all().all()):
    print(
        "Todas las variables son NaN. La celda ERA5-Land de este punto está en agua; "
        "cambiá LAT/LON a tierra firme y volvé a correr desde la celda de configuración."
    )

lat = float(frame["latitude"].dropna().iloc[0])
lon = float(frame["longitude"].dropna().iloc[0])
lon_cmp = lon - 360 if lon > 180 else lon
print(f"punto: lat={lat:.4f}, lon={lon_cmp:.4f}")
assert abs(lat - settings.lat) < 0.2
assert abs(lon_cmp - settings.lon) < 0.2